First, create the model. This must match the model used in the interactive training notebook.

In [1]:
from jetcam.csi_camera import CSICamera

from jetracer.nvidia_racecar import NvidiaRacecar

import cv2

import torch

import torchvision

CATEGORIES = ['apex']

device = torch.device('cuda')
model = torchvision.models.resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2 * len(CATEGORIES))
model = model.cuda().eval().half()

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


Next, load the saved model.  Enter the model path you used to save.

In [2]:
model.load_state_dict(torch.load('road_following_wide_model.pth'))

<All keys matched successfully>

Convert and optimize the model using ``torch2trt`` for faster inference with TensorRT.  Please see the [torch2trt](https://github.com/NVIDIA-AI-IOT/torch2trt) readme for more details.

> This optimization process can take a couple minutes to complete. 

In [3]:
from torch2trt import torch2trt

data = torch.zeros((1, 3, 126, 224)).cuda().half()
model_trt = torch2trt(model, [data], fp16_mode=True)

Save the optimized model using the cell below

In [4]:
torch.save(
    model_trt.state_dict(),
    'road_following_wide_trt.pth'
)

Load the optimized model by executing the cell below

In [1]:

from jetcam.csi_camera import CSICamera

from jetracer.nvidia_racecar import NvidiaRacecar

import cv2

import torch

import torchvision
from torch2trt import torch2trt

from torch2trt import TRTModule

model_trt = TRTModule()

model_trt.load_state_dict(
    torch.load('road_following_wide_trt.pth')
)

WARNNIG: Jetson.GPIO library has not been verified with this carrier board,


<All keys matched successfully>

Create the racecar class

In [2]:
from jetracer.nvidia_racecar import NvidiaRacecar

car = NvidiaRacecar()

Create the camera class.

In [3]:
from jetcam.csi_camera import CSICamera

camera = CSICamera(
    width=640,
    height=360,
    capture_width=1280,
    capture_height=720,
    capture_fps=30
)

camera.running = False

In [4]:
frame = camera.read()

print("Camera shape:", frame.shape)

Camera shape: (360, 640, 3)


In [5]:
from utils import preprocess

image = preprocess(frame).half()

print("Model input:", image.shape)

Model input: torch.Size([1, 3, 360, 640])


In [6]:
car.steering_offset = 0.07

In [7]:
car.steering_gain = -0.9

Finally, execute the cell below to make the racecar move forward, steering the racecar based on the x value of the apex.

Here are some tips,

* If the car wobbles left and right,  lower the steering gain
* If the car misses turns,  raise the steering gain
* If the car tends right, make the steering bias more negative (in small increments like -0.05)
* If the car tends left, make the steering bias more postive (in small increments +0.05)

In [12]:
from utils import preprocess
import numpy as np

STEERING_GAIN = 1
STEERING_BIAS = 0.00

car.throttle = 0.12

while True:
    image = camera.read()
    image = preprocess(image).half()
    output = model_trt(image).detach().cpu().numpy().flatten()
    x = float(output[0])
    car.steering = x * STEERING_GAIN + STEERING_BIAS

KeyboardInterrupt: 

In [8]:

import cv2
import torch
import threading
import ipywidgets as widgets
from IPython.display import display
from utils import preprocess

# Settings
camera.running = False
model_trt.eval()
car.throttle = 0.0

gain = widgets.FloatSlider(value=1.0, min=-2, max=2, step=0.05,
                           description='Steering Gain')
bias = widgets.FloatSlider(value=0.0, min=-1, max=1, step=0.01,
                           description='Steering Bias')
throttle = widgets.FloatSlider(value=0.12, min=0, max=0.25, step=0.01,
                               description='Throttle')

start_btn = widgets.Button(description='START', button_style='success')
stop_btn = widgets.Button(description='STOP', button_style='danger')
status = widgets.Label(value='Stopped')
camera_widget = widgets.Image(format='jpeg', width=448)

running = threading.Event()
car_lock = threading.Lock()
worker = None
for i in range(100):
    frame = camera.read()

print("Successfully captured 100 frames")

def drive_loop():
    try:
        count = 0
        while running.is_set():
            # Camera and inference
            frame = camera.read()

            # Resize to the trained model input
            small_frame = cv2.resize(
                frame,
                (224, 126),
                interpolation=cv2.INTER_AREA
            )

            with torch.no_grad():
                image = preprocess(small_frame).half()
                output = model_trt(image).detach().cpu().numpy().flatten()

            x = float(output[0])
            steering = max(-1.0, min(1.0, x * gain.value + bias.value))

            with car_lock:
                if not running.is_set():
                    break
                car.steering = steering
                car.throttle = throttle.value

            # Update preview every 3 frames
            if count % 3 == 0:
                view = frame.copy()
                h, w = view.shape[:2]

                # Predicted direction
                px = int(max(0, min(w - 1, (x + 1) * w / 2)))
                cv2.line(view, (w//2, h-1), (px, h//2), (0,255,0), 2)
                cv2.circle(view, (px, h//2), 5, (0,0,255), -1)

                cv2.putText(view, "Prediction: %.2f" % x,
                            (5, 20), cv2.FONT_HERSHEY_SIMPLEX,
                            0.45, (255,255,255), 1)

                camera_widget.value = cv2.imencode('.jpg', view)[1].tobytes()

            count += 1

    except Exception as e:
        status.value = "Error: {}".format(e)
    finally:
        with car_lock:
            running.clear()
            car.throttle = 0.0
            car.steering = 0.0
        if status.value == 'Running':
            status.value = 'Stopped'

def start_car(b):
    global worker
    if worker is not None and worker.is_alive():
        return
    running.set()
    status.value = 'Running'
    worker = threading.Thread(target=drive_loop, daemon=True)
    worker.start()

def stop_car(b):
    with car_lock:
        running.clear()
        car.throttle = 0.0
        car.steering = 0.0
    status.value = 'Stopped'

start_btn.on_click(start_car)
stop_btn.on_click(stop_car)

display(widgets.VBox([
    camera_widget,
    widgets.HBox([start_btn, stop_btn]),
    gain,
    bias,
    throttle,
    status
]))


Successfully captured 100 frames


In [9]:
if 'running' in globals():
    running.clear()
if 'worker' in globals() and worker is not None and worker.is_alive():
    worker.join(timeout=3.0)
    if worker.is_alive():
        raise RuntimeError("Previous driving loop is still running. Stop it first.")